# Семинар · Блок 1 — Квартиры на продажу в Бишкеке

**Образцовый разбор.** Здесь мы проходим один срез данных — *объявления о продаже
квартир в Бишкеке* — и применяем к нему всё, что уже изучили: распределения и
гистограммы, среднее / мода / разброс, выбросы, нормирование (цена за м² и
z-стандартизация), объект как точку и поиск похожих (kNN).


## 0. Инструменты и данные

In [ ]:
# В Google Colab сначала выполни:  !pip install -q datasets huggingface_hub
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from huggingface_hub import hf_hub_download

sns.set_theme(style="whitegrid")            # сетка на всех графиках
plt.rcParams["figure.dpi"] = 110
pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 160)

Скачиваем таблицу объявлений `listings` (фото не трогаем).

In [ ]:
path = hf_hub_download("aiacademy-kg/house_kg_full_dataset",
                       "data/listings.parquet", repo_type="dataset")
raw = pd.read_parquet(path)
print("строк:", len(raw), " колонок:", raw.shape[1])
raw.head(3)

## 1. Фокусируемся: Бишкек · продажа · квартиры

Одна строка — это **объявление** (не квартира: одну и ту же квартиру могут выложить
разные агентства — но это тема Блока 2). Сузимся до нашего среза.

In [ ]:
work = raw[(raw.deal == "sale") &
           (raw.type == "apartment") &
           (raw.city == "Бишкек")].copy()
print("объявлений в срезе:", len(work))

## 2. Чистка: лишние колонки и пропуски

В таблице 72 поля, большинство нам не нужны и во многих — сплошные пропуски
(поля для участков, коммуникаций, аренды…). Сначала посмотрим, **насколько заполнены**
колонки, и оставим только осмысленные для квартиры.

Оставляем компактный набор: идентификаторы, цена, площадь, комнаты, просмотры,
состояние/серия/этаж, координаты и текст объявления. Остальное — выкидываем.

In [ ]:
work.info()

In [ ]:
keep = ["id", "house_kg_id", "title", "address",
        "price_usd", "price_kgs", "area_m2", "rooms_n", "views",
        "condition", "building_series", "floor",
        "latitude", "longitude", "posted_date"]
work = work[keep]
work.head(3)

### 2.1 Пропуски в числе комнат — это не случайность

У части объявлений не распарсилось число комнат. Прежде чем просто их выкинуть —
посмотрим, **кто** эти объявления. Они не «обычные»: среди них аномально большие
площади (пентхаусы, «свободная планировка», ошибки ввода).

In [ ]:
miss = work[work.rooms_n.isna()].copy()
miss["area_m2"] = pd.to_numeric(miss["area_m2"], errors="coerce")
print("объявлений без числа комнат:", len(miss))
print("их площадь: median = %.0f  max = %.0f м²"
      % (miss.area_m2.median(), miss.area_m2.max()))
print("\nпримеры (заголовок · площадь):")
print(miss.sort_values("area_m2", ascending=False)[["title", "area_m2"]].head(8).to_string(index=False))

Вывод: пропуск в `rooms_n` часто маркирует **нетипичный** объект. На этом семинаре
мы честно их **отбрасываем** (не «додумываем» число комнат) — но теперь знаем, что
именно теряем. Заодно выкидываем строки без цены/площади.

In [ ]:
work["price_usd"] = pd.to_numeric(work["price_usd"], errors="coerce")
work["area_m2"]   = pd.to_numeric(work["area_m2"],   errors="coerce")
work["views"]     = pd.to_numeric(work["views"],     errors="coerce")

before = len(work)
work = work.dropna(subset=["price_usd", "area_m2", "rooms_n"]).copy()
print(f"было {before} → стало {len(work)} (отброшено {before - len(work)})")

## 3. Обзорные графики — сначала посмотреть на форму

Среднее и разброс — потом. Сначала глаза: как распределены цена, площадь и цена за м².

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
sns.histplot(work.price_usd, bins=60, kde=True, ax=ax[0]); ax[0].set_title("Цена, USD")
sns.histplot(work.area_m2,  bins=60, kde=True, ax=ax[1]); ax[1].set_title("Площадь, м²")
work_ppm2_preview = work.price_usd / work.area_m2
sns.histplot(work_ppm2_preview.clip(upper=6000), bins=60, kde=True, ax=ax[2]); ax[2].set_title("Цена за м², USD")
fig.tight_layout(); plt.show()

Все три — **право-скошенные** с длинным хвостом: основная масса слева, редкие
огромные значения тянут хвост вправо. По такой картинке уже видно, что среднее будет
завышено относительно «типичного» жилья.

In [ ]:
work.columns

# Coords

In [ ]:
sns.scatterplot(data=work, y='latitude', x='longitude')

### 3.1 Спойлер (просто пусть будет): логарифм выпрямляет хвост

Забегая вперёд — если взять **логарифм** цены и площади, тяжёлый хвост превращается в
аккуратный симметричный горб. Пока просто полюбуемся; зачем это — узнаем позже.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
sns.histplot(np.log10(work.price_usd), bins=60, ax=ax[0]); ax[0].set_title("log10(Цена)")
sns.histplot(np.log10(work.area_m2),  bins=60, ax=ax[1]); ax[1].set_title("log10(Площадь)")
fig.tight_layout(); plt.show()

### 3.2 Комнаты и структура облака

Число комнат — дискретное. А scatter «площадь × цена», раскрашенный по комнатам,
показывает, что общее облако — это **склеенные группы** (однушки, двушки, трёшки…).

In [ ]:
samp = work.sample(min(4000, len(work)), random_state=0).copy()
samp["rooms"] = samp.rooms_n.astype(int)
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
sns.countplot(x=work.rooms_n.astype(int), ax=ax[0]); ax[0].set_title("Число комнат")
sns.scatterplot(data=samp, x="area_m2", y="price_usd", hue="rooms",
                palette="viridis", s=14, ax=ax[1])
ax[1].set_title("Площадь × Цена (цвет — комнаты)")
ax[1].set_xlim(0, 300); ax[1].set_ylim(0, 600000)
fig.tight_layout(); plt.show()